# Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/cleaned_data.csv")
df.head()

,account_id,month,company_size,industry,contract_type,discount_pct,regime_state,active_users,usage_growth,feature_adoption_rate,error_rate,tickets_count,ticket_growth,payment_delay_flag,current_mrr,next_month_mrr
0,0,1,SMB,Energy,Monthly,0.054167,stable,4,0.017574,0.002766,0.102689,0,0.0,0,112.703497,50.000000
1,0,2,SMB,Energy,Monthly,0.054167,stable,11,0.027244,0.000000,0.102174,2,0.0,1,50.000000,51.903945
2,0,3,SMB,Energy,Monthly,0.054167,decline,12,0.004655,0.082195,0.195298,2,0.0,0,51.903945,50.000000
3,0,4,SMB,Energy,Monthly,0.054167,decline,17,-0.024496,0.062898,0.104958,3,0.5,1,50.000000,55.680067
4,0,5,SMB,Energy,Monthly,0.054167,decline,20,-0.073376,0.120356,0.076803,0,-1.0,0,55.680067,50.000000


# Sort Values

In [2]:
df = df.sort_values(["account_id", "month"])

# Create Targets

In [3]:
df["next_mrr"] = df.groupby("account_id")["current_mrr"].shift(-1)

df["churn"] = (
    df["next_mrr"] < df["current_mrr"] * 0.5
).astype(int)

df["churn"].value_counts(normalize=True)

churn
0    0.946556
1    0.053444
Name: proportion, dtype: float64

# 1. Feature Engineering

## 1.1 LAG Features

In [4]:
base_cols = [
    "current_mrr",
    "active_users",
    "usage_growth",
    "tickets_count",
    "error_rate"
]

for col in base_cols:
    df[f"{col}_lag1"] = df.groupby("account_id")[col].shift(1)
    df[f"{col}_lag2"] = df.groupby("account_id")[col].shift(2)

## 1.2 Growth Features

In [5]:
df["mrr_growth"] = df["current_mrr"] - df["current_mrr_lag1"]
df["usage_growth_change"] = df["usage_growth"] - df["usage_growth_lag1"]

## 1.3 Customer Age

In [6]:
df["customer_age"] = (
    df["month"] - df.groupby("account_id")["month"].transform("min")
)

## 1.4 Risk Features

In [7]:
df["high_error"] = (df["error_rate"] > 0.1).astype(int)
df["high_tickets"] = (df["tickets_count"] > 5).astype(int)

df["support_pressure"] = df["tickets_count"] / (df["active_users"] + 1)

# 2. Encoding

## 2.1 Encode Categoricals

### Encoding

In [8]:
cat_cols = ["company_size", "industry", "contract_type", "regime_state"]

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

### Features List

In [9]:
features = [
    col for col in df.columns
    if col not in ["account_id", "month", "churn", "next_mrr"]
]

In [27]:
features = [
    col for col in features
    if col not in ["next_month_mrr"]
]

## 2.2 Time Split

In [29]:
train = df[df["month"] <= 11]
val   = df[df["month"] > 11]

## 2.3 Datasets

In [30]:
X_train = train[features]
X_val = val[features]

y_train = train["churn"]
y_val = val["churn"]

In [31]:
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))

churn
0    0.907727
1    0.092273
Name: proportion, dtype: float64
churn
0    0.982148
1    0.017852
Name: proportion, dtype: float64


## 2.4 Imbalance

In [32]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 9.83743842364532


# 3. Modeling

## 3.1 Churn Model (Classification)

### Train Model

In [33]:
from lightgbm import LGBMClassifier

churn_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

churn_model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 50750, number of negative: 499250
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.030030 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4577
[LightGBM] [Info] Number of data points in the train set: 550000, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.092273 -> initscore=-2.286195
[LightGBM] [Info] Start training from score -2.286195


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### Predictions

In [34]:
val["churn_prob"] = churn_model.predict_proba(X_val)[:, 1]

### AUC

In [35]:
from sklearn.metrics import roc_auc_score

print("AUC:", roc_auc_score(y_val, val["churn_prob"]))

AUC: 0.9688421699996865


### Classification

In [36]:
from sklearn.metrics import classification_report

val["churn_pred"] = (val["churn_prob"] > 0.3).astype(int)
print(classification_report(y_val, val["churn_pred"]))

              precision    recall  f1-score   support

           0       1.00      0.97      0.98    589289
           1       0.33      0.87      0.48     10711

    accuracy                           0.97    600000
   macro avg       0.66      0.92      0.73    600000
weighted avg       0.99      0.97      0.97    600000



## 3.2 MRR Model (Regression)

In [37]:
train_mrr = train.dropna(subset=["next_mrr"])
val_mrr = val.dropna(subset=["next_mrr"])

### Train

In [38]:
from lightgbm import LGBMRegressor

X_train_mrr = train_mrr[features]
y_train_mrr = train_mrr["next_mrr"]

X_val_mrr = val_mrr[features]
y_val_mrr = val_mrr["next_mrr"]

mrr_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

mrr_model.fit(X_train_mrr, y_train_mrr)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045190 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4577
[LightGBM] [Info] Number of data points in the train set: 550000, number of used features: 39
[LightGBM] [Info] Start training from score 278.810901


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### Evaluation

In [39]:
from sklearn.metrics import mean_absolute_error

val_mrr["mrr_pred"] = mrr_model.predict(X_val_mrr)
mae = mean_absolute_error(y_val_mrr, val_mrr["mrr_pred"])

print("MAE:", mae)
print("Error %:", mae / y_val_mrr.mean())

MAE: 70.18076533018278
Error %: 0.24649798989393024


### Baseline Comparison

In [40]:
print("Mean MRR:", y_train_mrr.mean())

Mean MRR: 278.8109014521647


## 3.3 Importance

In [41]:
importance = pd.Series(
    churn_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance.head(10))

current_mrr              728
feature_adoption_rate    623
mrr_growth               591
active_users             508
support_pressure         467
ticket_growth            465
current_mrr_lag1         464
usage_growth             419
tickets_count            412
usage_growth_lag2        367
dtype: int32


## 3.4 Business Metric

In [43]:
val = val.merge(
    val_mrr[["account_id", "month", "mrr_pred"]],
    on=["account_id", "month"],
    how="left"
)

### Top Customers

In [44]:
y_train_log = np.log1p(y_train_mrr)
mrr_model.fit(X_train, y_train_log)

val["mrr_pred"] = np.expm1(mrr_model.predict(X_val))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034617 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4577
[LightGBM] [Info] Number of data points in the train set: 550000, number of used features: 39
[LightGBM] [Info] Start training from score 4.532768


In [45]:
val["expected_loss"] = val["churn_prob"] * val["mrr_pred"]

In [46]:
val.sort_values("expected_loss", ascending=False).head(10)

,account_id,month,discount_pct,active_users,usage_growth,feature_adoption_rate,error_rate,tickets_count,ticket_growth,payment_delay_flag,...,industry_SaaS,contract_type_Monthly,regime_state_growth,regime_state_stable,churn_prob,churn_pred,mrr_pred_x,mrr_pred_y,mrr_pred,expected_loss
82436,6869,20,0.061534,332,0.027189,0.943744,0.043405,32,0.777778,1,...,False,False,True,False,0.429605,1,14310.338030,14310.338030,14878.788575,6392.003611
37235,3102,23,0.036357,459,0.037822,0.000000,0.105370,35,0.842105,1,...,False,False,True,False,0.370763,1,NaN,NaN,13225.195765,4903.409557
205625,17135,17,0.073043,294,0.038613,0.816410,0.082160,14,0.750000,1,...,False,False,True,False,0.295272,0,22544.793582,22544.793582,16055.419010,4740.714847
139215,11601,15,0.186253,160,0.038172,0.715551,0.125828,3,-0.571429,0,...,False,True,False,True,0.749925,1,6235.471466,6235.471466,6038.064029,4528.096892
46246,3853,22,0.000000,439,0.084964,1.000000,0.098517,37,0.608696,0,...,False,False,True,False,0.297829,0,21346.799745,21346.799745,14246.743048,4243.086618
570381,47531,21,0.048952,382,0.078135,0.000000,0.136368,18,0.500000,1,...,True,False,True,False,0.319939,1,18613.136042,18613.136042,12373.781779,3958.850857
264826,22068,22,0.087104,376,0.032396,1.000000,0.085115,24,0.043478,1,...,False,False,False,True,0.217011,0,18473.546577,18473.546577,17203.660584,3733.385112
260427,21702,15,0.044441,275,0.067086,0.686617,0.089488,14,0.750000,1,...,False,False,True,False,0.215685,0,24056.199180,24056.199180,16532.026374,3565.705448
26508,2209,12,0.048401,223,0.003760,0.566963,0.104628,15,0.875000,0,...,False,True,False,True,0.267932,0,14059.593933,14059.593933,13146.337457,3522.322428
58792,4899,16,0.028444,281,-0.010675,0.835365,0.094420,17,0.545455,1,...,False,False,False,False,0.206557,0,22265.820779,22265.820779,15697.611671,3242.455593


## 3.5 Desicion Rules

### Segmentation

In [47]:
def segment(row):
    if row["churn_prob"] > 0.4 and row["mrr_pred"] > 300:
        return "High Risk - High Value"
    elif row["churn_prob"] > 0.4:
        return "High Risk - Low Value"
    else:
        return "Safe"

val["segment"] = val.apply(segment, axis=1)

print(val["segment"].value_counts())

segment
Safe                      576073
High Risk - Low Value      22545
High Risk - High Value      1382
Name: count, dtype: int64


### Revenue Dashboard

In [49]:
print("Total Revenue at Risk:", val["expected_loss"].sum())
print("Average Risk per Customer:", val["expected_loss"].mean())

Total Revenue at Risk: 6502212.238498804
Average Risk per Customer: 10.837020397498007


### Top Customers Action List

In [50]:
top_customers = val.sort_values("expected_loss", ascending=False).head(20)
top_customers[["account_id", "churn_prob", "mrr_pred", "expected_loss"]]

,account_id,churn_prob,mrr_pred,expected_loss
82436,6869,0.429605,14878.788575,6392.003611
37235,3102,0.370763,13225.195765,4903.409557
205625,17135,0.295272,16055.419010,4740.714847
139215,11601,0.749925,6038.064029,4528.096892
46246,3853,0.297829,14246.743048,4243.086618
570381,47531,0.319939,12373.781779,3958.850857
264826,22068,0.217011,17203.660584,3733.385112
260427,21702,0.215685,16532.026374,3565.705448
26508,2209,0.267932,13146.337457,3522.322428
58792,4899,0.206557,15697.611671,3242.455593
